In [ ]:
import sqlite3
from sqlite3 import Error

## Função de alteração (insert, update, delete...)
def execute_query(connection, query):
    cursor = connection.cursor()
    try:
        cursor.execute(query)
        connection.commit()
        print(f"Query {query.split('(')[0].strip()} executada.")
        if cursor.rowcount != -1: print(f"{cursor.rowcount} linha(s) afetadas")

    except Error as e:
        print(f"Erro: '{e}'")


# Função de leitura de dados (select)
def execute_read_query(connection, query):
    cursor = connection.cursor()
    result = None
    try:
        cursor.execute(query)
        result = cursor.fetchall()

        return result
    except Error as e:
        print(f"Erro: '{e}'")

## Atividade 01
Crie duas consultas relevantes para sua aplicação em PI3 que tenham, pelo menos, uma função de agregação em cada. Para cada consulta, descreva a ação esperada (ex.: contar participantes em um evento) e justifique o contexto na aplicação de PI3 (ex.: funcionalidade do back-end para checar se já atingiu o limite de vagas do evento).



**CONSULTA 1**

* Objetivo da consulta (contexto de PI3): Contar a quantidade total de disciplinas cadastradas que estão associadas a cada professor, permitindo que o painel administrativo exiba a carga de trabalho de cada docente.
* Descrição da ação esperada ao rodar a consulta: Agrupar as disciplinas pelo ID e nome do professor e retornar o total de disciplinas vinculadas a cada um deles.


In [ ]:
SELECT p.id, p.nome AS professor, COUNT(d.id) AS total_disciplinas
FROM professor p
LEFT JOIN disciplina d ON p.id = d.id_professor
GROUP BY p.id, p.nome;


**CONSULTA 2**

* Objetivo da consulta (contexto de PI3):Identificar a média ou a quantidade de matérias/arquivos de estudo cadastrados por disciplina para monitorar quais conteúdos possuem mais recursos disponíveis.
* Descrição da ação esperada ao rodar a consulta: Contar quantas matérias estão cadastradas para cada disciplina específica.

In [ ]:
SELECT d.nome AS disciplina, COUNT(m.id) AS total_materias
FROM disciplina d
LEFT JOIN materia m ON d.id = m.id_disciplina
GROUP BY d.id, d.nome;

## Atividade 02

Crie uma consulta relevante para sua aplicação em PI3 que envolva três entidades distintas. Descreva a ação esperada e justifique o contexto na aplicação de PI3.


**CONSULTA COM 3 ENTIDADES**

* Objetivo da consulta (contexto de PI3): Exibir a relação completa das disciplinas que os usuários (alunos) estão matriculados, juntamente com o nome do professor responsável por cada uma dessas disciplinas.
* Descrição da ação esperada ao rodar a consulta: Cruzar as tabelas de usuário (usuario), a tabela associativa de matrículas (usuario_disciplina), a tabela de disciplinas (disciplina) e a tabela de professores (professor) para listar o nome do aluno, a disciplina cursada e o respectivo professor.

In [ ]:
SELECT u.nome AS aluno, d.nome AS disciplina, p.nome AS professor
FROM usuario u
JOIN usuario_disciplina ud ON u.id = ud.id_usuario
JOIN disciplina d ON ud.id_disciplina = d.id
JOIN professor p ON d.id_professor = p.id;

## Atividade 03

Crie uma view relevante para sua aplicação em PI3. Descreva a ação esperada na consulta dentro de sua view e, ao justificar o contexto, informe também por que a ação é tão recorrente no seu PI3 a ponto de precisar de uma view.


* Justificativa para que a consulta seja uma View (Contexto de PI3): A listagem detalhada de disciplinas com seus respectivos professores é exigida em várias telas do sistema (como na página inicial, na listagem pública de cursos e no painel do aluno). Criar uma view evita a repetição constante desse bloco complexo de JOIN.
* Descrição da ação esperada ao rodar a consulta: Retornar uma tabela virtual consolidada contendo o identificador e nome da disciplina, sua descrição e o nome completo do professor responsável.

In [ ]:
-- Criação da View
CREATE VIEW vw_disciplinas_professores AS
SELECT
    d.id AS id_disciplina,
    d.nome AS nome_disciplina,
    d.descricao AS descricao_disciplina,
    p.id AS id_professor,
    p.nome AS nome_professor,
    p.email AS email_professor
FROM disciplina d
LEFT JOIN professor p ON d.id_professor = p.id;

In [ ]:
-- Select que retorna todos os elementos da View
SELECT * FROM vw_disciplinas_professores;

## Atividade 04

Crie um procedure relevante para sua aplicação em PI3. Descreva a ação esperada ao executar a procedure e, ao justificar o contexto, explique por que essa ação é tão recorrente no seu PI3 a ponto de precisar de uma procedure.


* Justificativa para que a operação seja um Procedure (Contexto de PI3): A associação de um usuário (aluno) a uma disciplina é uma transação recorrente na rotina acadêmica. Encapsular essa lógica em uma procedure garante validação e segurança no backend ao processar matrículas.
* Descrição da ação esperada ao executar o procedure: Inserir um novo registro na tabela associativa usuario_disciplina, vinculando o ID do usuário informado ao ID da disciplina informada.

In [ ]:
-- Criação da Procedure (Sintaxe compatível com MySQL/MariaDB)
DELIMITER //
CREATE PROCEDURE sp_matriar_usuario_disciplina(
    IN p_id_usuario INT,
    IN p_id_disciplina INT
)
BEGIN
    INSERT INTO usuario_disciplina (id_usuario, id_disciplina)
    VALUES (p_id_usuario, p_id_disciplina);
END //
DELIMITER ;

## Atividade 05

Crie uma function relevante para sua aplicação em PI3.
Descreva o valor esperado como retorno ao executar a function e, ao justificar o contexto, explique por que é importante ter essa lógica encapsulada para uso recorrente no seu PI3.


* Justificativa para que a operação seja uma Function (Contexto de PI3): É importante calcular de forma rápida e dinâmica quantas disciplinas um determinado professor leciona diretamente em consultas de relatórios, sem precisar reescrever a mesma subconsulta de contagem repetidas vezes.
* Descrição da operação realizada e valor retornado pela Function: A função recebe o ID de um professor e retorna um valor inteiro (INT) correspondente à quantidade de disciplinas atreladas a ele.

In [ ]:
-- Criação da Function
DELIMITER //
CREATE FUNCTION fn_total_disciplinas_professor(p_id_professor INT)
RETURNS INT
DETERMINISTIC
BEGIN
    DECLARE total INT;
    SELECT COUNT(*) INTO total
    FROM disciplina
    WHERE id_professor = p_id_professor;
    RETURN total;
END //
DELIMITER ;

In [ ]:
-- Consulta que inclui a Function
SELECT id, nome, fn_total_disciplinas_professor(id) AS qtd_disciplinas
FROM professor;

## Atividade 06

Crie um trigger relevante para sua aplicação em PI3.
Descreva o evento que aciona o trigger e, ao justificar o contexto, explique por que essa automação é necessária para manter a integridade ou eficiência do seu PI3.


* Justificativa para que a operação seja um Trigger (Contexto de PI3): Para fins de auditoria ou segurança, pode ser necessário registrar logs ou impedir exclusões acidentais de professores que possuem disciplinas ativas vinculadas, mantendo a integridade referencial e lógica do sistema. (Nota: Como o banco já possui ON DELETE CASCADE na FK, este trigger ilustra uma validação preventiva de bloqueio antes de deletar um professor caso ele possua disciplinas).
* Descrição do evento que dispara o Trigger e das operações que ele executa: O gatilho é acionado BEFORE DELETE na tabela professor. Ele verifica se o professor possui disciplinas cadastradas; caso possua, cancela a operação ou impede a exclusão para evitar inconsistências.

In [ ]:
-- Criação do Trigger
DELIMITER //
CREATE TRIGGER trg_impede_exclusao_professor
BEFORE DELETE ON professor
FOR EACH ROW
BEGIN
    DECLARE qtd INT;
    SELECT COUNT(*) INTO qtd FROM disciplina WHERE id_professor = OLD.id;

    IF qtd > 0 THEN
        SIGNAL SQLSTATE '45000'
        SET MESSAGE_TEXT = 'Erro: Não é possível excluir um professor que possui disciplinas atreladas.';
    END IF;
END //
DELIMITER ;

In [ ]:
-- Comando SQL que dispara o Trigger (tentar excluir o professor de ID 1, que possui disciplinas)
DELETE FROM professor WHERE id = 1;